In [1]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\admin\AppData\Local\Temp\ipykernel_8908\2148886914.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [3]:
DATA_DIR = Path("data")

In [4]:
persist_directory = DATA_DIR / "bookstore"

In [5]:
pdf_files_paths = DATA_DIR.glob("*.pdf")

In [6]:
list(pdf_files_paths)

[WindowsPath('data/Atomic habits ( PDFDrive ).pdf'),
 WindowsPath('data/attention.pdf'),
 WindowsPath('data/BhagavadGita.pdf'),
 WindowsPath('data/the-5-am-club.pdf')]

In [7]:
def normalize_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text.lower()).strip("_")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()

In [8]:
all_documents = []
source_ids = set() 

for pdf_file_path in DATA_DIR.glob("*.pdf"):
    print(f"{pdf_file_path.name}: {file_sha256(pdf_file_path)}")
    loader = PyMuPDFLoader(str(pdf_file_path))
    documents = loader.load()

    source_checksum = file_sha256(pdf_file_path)
    source_id = normalize_name(pdf_file_path.stem)
    source_ids.add(source_id)

    for index, doc in enumerate(documents):
        doc.metadata["source"] = pdf_file_path.name
        doc.metadata["source_id"] = source_id
        doc.metadata["source_checksum"] = source_checksum
        doc.metadata["page_number"] = index + 1
        doc.metadata["chunk_id"] = str(uuid4())
        doc.metadata["chunk_index"] = index
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["created_at"] = datetime.now(timezone.utc).isoformat()

    all_documents.extend(documents)

documents = all_documents
print(len(documents), "documents loaded from all PDFs.")
sorted(set(source_ids))

Atomic habits ( PDFDrive ).pdf: a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878
attention.pdf: bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697
BhagavadGita.pdf: ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583
the-5-am-club.pdf: 089bb3fc5b41e7de713444b93214434e1d887c47dff66e8e4ae075aeed89440b
1475 documents loaded from all PDFs.


['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

In [10]:
chunks = splitter.split_documents(documents)
chunks[0].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits \\( PDFDrive.com \\).pdf',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 2,
 'source_id': 'atomic_habits___pdfdrive',
 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'page_number': 3,
 'chunk_id': '013be760-2336-4699-9520-7a19309d438d',
 'chunk_index': 2,
 'chunk_size': 531,
 'created_at': '2026-07-28T19:35:43.227935+00:00'}

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
collection_name = "bookstore"
collection_name

['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [ ]:
if not persist_directory.exists():
    persist_directory.mkdir(parents=True, exist_ok=True)

temp_store = Chroma(
    persist_directory=str(persist_directory),
    embedding_function=embeddings,
    collection_name=collection_name,
    )

try:
    temp_store._client.delete_collection(name=collection_name)
except Exception:
    pass

vector_store = Chroma(
    persist_directory=str(persist_directory),
    embedding_function=embeddings,
    collection_name=collection_name,
    )

vector_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x235e54cb140>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x235e8d84530>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x235e8db3da0>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x235e8e0fe90>}

In [ ]:
inserted_count = 0
batch_size = 100

for start in range(0, len(chunks), batch_size):
    end = start + batch_size
    vector_store.add_documents(chunks[start:end])
    inserted_count += len(chunks[start:end])

print("Inserted chunks:", inserted_count)

NotFoundError: Error in compaction: Error getting collection with segments: Collection [001f5d20-b9f6-4e90-b316-4218b56212d3] does not exist.

# Creating retriever for the RAG system

In [ ]:
query = ['What does krishna says about the Mahabharat battle']

In [ ]:
vector_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x1f4a101fa40>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1f4a0ef6240>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1f4a0c0db20>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x1f4a105cda0>}

In [ ]:
bg_vectorstore = vector_store
bg_vectorstore

In [ ]:
bg_retriver = bg_vectorstore.as_retriever(
    search_kwargs={"k": 4, "filter": {"source_id": "bhagavadgita"}}
)

In [ ]:
bg_retriver.invoke(input=query[0])

[Document(id='d3e3b67c-981d-4454-b02c-27e94d81b666', metadata={'modDate': "D:20120508142201Z00'00'", 'total_pages': 952, 'chunk_id': '2e466f3f-1296-45ea-96e4-ba12cceff96c', 'subject': '', 'title': 'Bhagavad-gita As It Is with pics!', 'source': 'BhagavadGita.pdf', 'creator': 'Pages', 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'creationdate': "D:20120508142201Z00'00'", 'format': 'PDF 1.3', 'chunk_index': 50, 'creationDate': "D:20120508142201Z00'00'", 'keywords': '', 'file_path': 'data\\BhagavadGita.pdf', 'created_at': '2026-07-28T19:15:57.103441+00:00', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'moddate': "D:20120508142201Z00'00'", 'chunk_size': 1333, 'source_id': 'bhagavadgita', 'page_number': 51, 'trapped': '', 'author': 'me', 'page': 50}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

# Getting Similarity Score

## How to read similarity score

`similarity_search_with_score()` returns a list of `(Document, score)`.

What the score means in this setup:
- It is a distance-like value from Chroma.
- Lower score = more similar match.
- Higher score = less similar match.

How to understand it correctly:
1. Rank by score ascending (smallest first).
2. Compare scores only within the same query and same collection.
3. Use a simple gap check:
   - if top score is much lower than the next one, top result is strongly preferred.
   - if scores are close, results are similarly relevant.

Practical note:
- Do not use one fixed cutoff blindly across all collections/embedding models.
- Start by observing score ranges on your own data and then set thresholds.

In [ ]:
bg_vectorstore = vector_store

bg_vectorstore.similarity_search_with_score(
    "What does krishna tell parth in the bnbattle ground",
    k=4,
    filter={"source_id": "bhagavadgita"},
)

[(Document(id='b63c6fc0-1c4e-44bb-89e3-7ac274de0a6f', metadata={'chunk_id': '2e466f3f-1296-45ea-96e4-ba12cceff96c', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'file_path': 'data\\BhagavadGita.pdf', 'moddate': "D:20120508142201Z00'00'", 'trapped': '', 'source': 'BhagavadGita.pdf', 'source_id': 'bhagavadgita', 'title': 'Bhagavad-gita As It Is with pics!', 'keywords': '', 'subject': '', 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'created_at': '2026-07-28T19:15:57.103441+00:00', 'format': 'PDF 1.3', 'page_number': 51, 'chunk_size': 1333, 'modDate': "D:20120508142201Z00'00'", 'creationDate': "D:20120508142201Z00'00'", 'creationdate': "D:20120508142201Z00'00'", 'author': 'me', 'chunk_index': 50, 'page': 50, 'total_pages': 952, 'creator': 'Pages'}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Suprem